In [1]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(3, 32)   # 3 input channels if you use 3 slices
        self.pool = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64)
        self.down3 = DoubleConv(64, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv2 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.pool(x1)
        x3 = self.down2(x2)
        x4 = self.pool(x3)
        x5 = self.down3(x4)

        x = self.up1(x5)
        x = torch.cat([x, x3], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv2(x)

        return self.out(x)  # no sigmoid here

C:\Users\Mohamed\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\Mohamed\anaconda3\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
C:\Users\Mohamed\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [9]:
import os

def build_dataset(root_path, subjects):
    data_list = []

    for subject in subjects:
        ct_path = os.path.join(root_path, subject, "CT")
        uct_path = os.path.join(root_path, subject, "UCT")

        ct_file = None
        mask_file = None

        # find files
        for f in os.listdir(ct_path):
            if "CT_RAW" in f:
                ct_file = os.path.join(ct_path, f)

        for f in os.listdir(uct_path):
            if f.endswith("_LABELS.nii"):
                mask_file = os.path.join(uct_path, f)

        if ct_file and mask_file:
            data_list.append({
                "subject": subject,
                "ct": ct_file,
                "mask": mask_file
            })

    return data_list

In [10]:
train_subjects = [f"F{i:02d}" for i in range(1, 11)] + [f"T{i:02d}" for i in range(1, 6)]

val_subjects = [f"F{i:02d}" for i in range(11, 14)]

test_subjects = [f"F{i:02d}" for i in range(14, 16)] + [f"T{i:02d}" for i in range(6, 9)]

In [11]:
root = "DATASET_ROOT"

train_data = build_dataset(root, train_subjects)
val_data = build_dataset(root, val_subjects)
test_data = build_dataset(root, test_subjects)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

Train: 15
Val: 3
Test: 5


In [12]:
import SimpleITK as sitk
import numpy as np
import nibabel as nib

def resample_mask_to_ct(ct_path, mask_path):
    ct = sitk.ReadImage(ct_path)
    mask = sitk.ReadImage(mask_path)

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(ct)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)  # IMPORTANT for labels

    mask_resampled = resampler.Execute(mask)

    return ct, mask_resampled


sample = train_data[0]

ct_img, mask_img = resample_mask_to_ct(sample["ct"], sample["mask"])

ct_arr = sitk.GetArrayFromImage(ct_img)
mask_arr = sitk.GetArrayFromImage(mask_img)

print("CT:", ct_arr.shape)
print("Mask:", mask_arr.shape)

CT: (97, 115, 95)
Mask: (97, 115, 95)


In [13]:
import torch
from torch.utils.data import Dataset
import nibabel as nib
import numpy as np
import torch.nn.functional as F

class CTDataset2D(Dataset):
    def __init__(self, data_list, size=(128, 128), min_mask_sum=50):
        self.size = size
        self.samples = []

        for item in data_list:
            ct = nib.load(item["ct"]).get_fdata()
            mask = nib.load(item["mask"]).get_fdata()

            depth = ct.shape[2]

            for i in range(depth):

                m = mask[:, :, i]

                # FILTER: skip empty slices
                if m.sum() < min_mask_sum:
                    continue

                self.samples.append((item, i))

    def __len__(self):
        return len(self.samples)

    def normalize(self, img):
        img = img.astype(np.float32)
        return (img - img.mean()) / (img.std() + 1e-8)

    def resize(self, img, is_mask=False):
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

        mode = "nearest" if is_mask else "bilinear"

        img = F.interpolate(
            img,
            size=self.size,
            mode=mode,
            align_corners=False if not is_mask else None
        )

        return img.squeeze(0).squeeze(0)

    def __getitem__(self, idx):
        item, i = self.samples[idx]

        ct = nib.load(item["ct"]).get_fdata()
        mask = nib.load(item["mask"]).get_fdata()

        depth = ct.shape[2]

        # 3 adjacent slices
        i0 = max(i - 1, 0)
        i1 = i
        i2 = min(i + 1, depth - 1)

        s0 = self.resize(self.normalize(ct[:, :, i0]), is_mask=False)
        s1 = self.resize(self.normalize(ct[:, :, i1]), is_mask=False)
        s2 = self.resize(self.normalize(ct[:, :, i2]), is_mask=False)

        # STACK → (3, H, W)
        ct_tensor = torch.stack([s0, s1, s2], dim=0)

        mask_slice = self.resize(mask[:, :, i], is_mask=True)
        mask_tensor = (mask_slice > 0.5).float().unsqueeze(0)

        return ct_tensor, mask_tensor

In [14]:
import SimpleITK as sitk
import os

def preprocess_subject(ct_path, mask_path, out_mask_path):
    ct = sitk.ReadImage(ct_path)
    mask = sitk.ReadImage(mask_path)

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(ct)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)

    mask_aligned = resampler.Execute(mask)

    sitk.WriteImage(mask_aligned, out_mask_path)


for item in train_data + val_data + test_data:
    ct_path = item["ct"]
    mask_path = item["mask"]

    out_mask = mask_path.replace(".nii", "_aligned.nii")

    preprocess_subject(ct_path, mask_path, out_mask)

    item["mask"] = out_mask

In [15]:
from torch.utils.data import DataLoader

train_dataset = CTDataset2D(train_data)
val_dataset = CTDataset2D(val_data)
test_dataset = CTDataset2D(test_data)


train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


print("Train samples:", len(train_dataset))

Train samples: 1108


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNetSmall().to(device)
model.load_state_dict(torch.load("unet_final.pth", map_location=device))
model.eval()

UNetSmall(
  (down1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (down2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (down3): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
  )
  (up1): ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
  (conv1): DoubleCon

In [16]:
def dice(pred, target, eps=1e-6):
    pred = (pred > 0.5).float()
    target = target.float()

    intersection = (pred * target).sum()
    return (2. * intersection + eps) / (pred.sum() + target.sum() + eps)

In [17]:
model.eval()
scores = []

with torch.no_grad():
    for ct, mask in val_loader:
        ct = ct.to(device)
        mask = mask.to(device)

        pred = model(ct)

        scores.append(dice(pred, mask).item())

print("Dice score:", sum(scores)/len(scores))

Dice score: 0.7747709545595892


In [18]:
model.eval()
test_scores = []

with torch.no_grad():
    for ct, mask in test_loader:
        ct = ct.to(device)
        mask = mask.to(device)

        logits = model(ct)
        probs = torch.sigmoid(logits)

        test_scores.append(dice(probs, mask).item())

print("Test Dice:", sum(test_scores) / len(test_scores))

Test Dice: 0.7563155929247538


In [19]:
with torch.no_grad():
    logits = model(ct)
    probs = torch.sigmoid(logits)

pred = (probs > 0.5).cpu().numpy()

# CLEAN STEP
import scipy.ndimage as nd

labeled, num = nd.label(pred)
clean = np.zeros_like(pred)

for i in range(1, num+1):
    component = (labeled == i)
    if component.sum() > 50:
        clean[component] = 1

In [20]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nibabel.affines import apply_affine

# ============================================================
# CONFIG
# ============================================================
ROOT = "DATASET_ROOT"

TRAIN_SUBJECTS = [f"F{i:02d}" for i in range(1, 11)] + [f"T{i:02d}" for i in range(1, 6)]
VAL_SUBJECTS   = [f"F{i:02d}" for i in range(11, 14)]
TEST_SUBJECTS  = [f"F{i:02d}" for i in range(14, 16)] + [f"T{i:02d}" for i in range(6, 9)]

LANDMARK_NAMES = ["RW", "C", "A", "OW", "V"]

# Network input/output size in (D, H, W)
# Keep these divisible by 4 because we use 2 pooling layers
TARGET_SHAPE = (96, 128, 128)

SIGMA = 2.5
BATCH_SIZE = 1
EPOCHS = 40
LR = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# DATA DISCOVERY
# ============================================================
def build_dataset(root_path, subjects):
    data_list = []

    for subject in subjects:
        subject_path = os.path.join(root_path, subject)
        ct_path = os.path.join(subject_path, "CT")

        ct_file = None
        desc_file = None

        if os.path.isdir(ct_path):
            for f in os.listdir(ct_path):
                if "CT_RAW" in f and f.endswith(".nii"):
                    ct_file = os.path.join(ct_path, f)
                    break

        if os.path.isdir(subject_path):
            for f in os.listdir(subject_path):
                if f.endswith("_DESC.json"):
                    desc_file = os.path.join(subject_path, f)
                    break

        if ct_file and desc_file:
            data_list.append({
                "subject": subject,
                "ct": ct_file,
                "desc": desc_file
            })

    return data_list


train_data = build_dataset(ROOT, TRAIN_SUBJECTS)
val_data   = build_dataset(ROOT, VAL_SUBJECTS)
test_data  = build_dataset(ROOT, TEST_SUBJECTS)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))


# ============================================================
# DESC LOADING
# ============================================================
def load_desc(desc_path):
    with open(desc_path, "r") as f:
        data = json.load(f)

    landmarks = {
        k: np.asarray(v, dtype=np.float32)
        for k, v in data["LANDMARKS"].items()
    }

    return {
        "id": data.get("ID", ""),
        "side": data.get("SIDE", ""),
        "landmarks": landmarks,
        "origin": np.asarray(data["COORD"]["ORIGIN"], dtype=np.float32),
        "xaxis": np.asarray(data["COORD"]["XAXIS"], dtype=np.float32),
        "yaxis": np.asarray(data["COORD"]["YAXIS"], dtype=np.float32),
        "zaxis": np.asarray(data["COORD"]["ZAXIS"], dtype=np.float32),
    }


# ============================================================
# HELPERS
# ============================================================
def normalize_volume(vol):
    vol = vol.astype(np.float32)
    return (vol - vol.mean()) / (vol.std() + 1e-8)


def world_to_voxel(world_coord, affine):
    inv_affine = np.linalg.inv(affine)
    return apply_affine(inv_affine, world_coord)


def voxel_xyz_to_net(v_xyz):
    # CT array is (X, Y, Z)
    # network uses (D, H, W) = (Z, X, Y)
    x, y, z = v_xyz
    return np.array([z, x, y], dtype=np.float32)


def net_to_voxel_xyz(v_net):
    # (D, H, W) -> (X, Y, Z)
    d, h, w = v_net
    return np.array([h, w, d], dtype=np.float32)


def resize_volume_3d(vol, target_shape):
    """
    vol: numpy array with shape (D, H, W)
    returns torch tensor with shape (1, D, H, W)
    """
    t = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=target_shape, mode="trilinear", align_corners=False)
    return t.squeeze(0)


def resize_heatmap_3d(hm, target_shape):
    """
    hm: numpy array with shape (D, H, W)
    returns torch tensor with shape (D, H, W)
    """
    t = torch.tensor(hm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=target_shape, mode="trilinear", align_corners=False)
    return t.squeeze(0).squeeze(0)


def make_gaussian_3d(shape, center, sigma=2.5):
    """
    shape: (D, H, W)
    center: (d, h, w)
    """
    D, H, W = shape
    dz = np.arange(D, dtype=np.float32)[:, None, None]
    dy = np.arange(H, dtype=np.float32)[None, :, None]
    dx = np.arange(W, dtype=np.float32)[None, None, :]

    cd, ch, cw = center
    dist2 = (dz - cd) ** 2 + (dy - ch) ** 2 + (dx - cw) ** 2
    hm = np.exp(-dist2 / (2 * sigma ** 2)).astype(np.float32)
    return hm


def decode_heatmap_argmax(hm):
    """
    hm: torch tensor or numpy array with shape (D, H, W)
    returns (d, h, w)
    """
    if torch.is_tensor(hm):
        idx = torch.argmax(hm).item()
        D, H, W = hm.shape
    else:
        idx = int(np.argmax(hm))
        D, H, W = hm.shape

    d = idx // (H * W)
    rem = idx % (H * W)
    h = rem // W
    w = rem % W
    return np.array([d, h, w], dtype=np.float32)


# ============================================================
# DATASET
# ============================================================
class LandmarkDataset(Dataset):
    def __init__(self, data_list, landmark_names, target_shape=(96, 128, 128), sigma=2.5):
        self.data_list = data_list
        self.landmark_names = landmark_names
        self.target_shape = target_shape
        self.sigma = sigma

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]

        ct_img = nib.load(item["ct"])
        ct = ct_img.get_fdata().astype(np.float32)
        affine = ct_img.affine

        # Original CT shape is (X, Y, Z)
        orig_shape_xyz = np.array(ct.shape, dtype=np.float32)

        # Convert to network order: (D, H, W) = (Z, X, Y)
        ct_net = np.transpose(ct, (2, 0, 1))
        ct_net = normalize_volume(ct_net)

        # Resize CT
        ct_resized = resize_volume_3d(ct_net, self.target_shape)

        # Build heatmaps
        desc = load_desc(item["desc"])
        heatmaps = []

        for name in self.landmark_names:
            world = desc["landmarks"][name]
            voxel_xyz = world_to_voxel(world, affine)
            voxel_net = voxel_xyz_to_net(voxel_xyz)

            # Scale voxel coords from original net shape -> target shape
            orig_net_shape = np.array([ct.shape[2], ct.shape[0], ct.shape[1]], dtype=np.float32)
            scale = (np.array(self.target_shape, dtype=np.float32) - 1.0) / (orig_net_shape - 1.0 + 1e-8)
            center = voxel_net * scale

            hm = make_gaussian_3d(self.target_shape, center=center, sigma=self.sigma)
            heatmaps.append(hm)

        heatmaps = np.stack(heatmaps, axis=0)  # (L, D, H, W)
        heatmaps = torch.tensor(heatmaps, dtype=torch.float32)

        return ct_resized, heatmaps, item["subject"], affine, ct.shape


# ============================================================
# MODEL
# ============================================================
class DoubleConv3D(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=5, base=16):
        super().__init__()
        self.enc1 = DoubleConv3D(in_channels, base)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = DoubleConv3D(base, base * 2)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(base * 2, base * 4)

        self.up2 = nn.ConvTranspose3d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv3D(base * 4, base * 2)

        self.up1 = nn.ConvTranspose3d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv3D(base * 2, base)

        self.out = nn.Conv3d(base, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.pool1(x1)

        x3 = self.enc2(x2)
        x4 = self.pool2(x3)

        x5 = self.bottleneck(x4)

        x = self.up2(x5)
        x = torch.cat([x, x3], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        return self.out(x)


# ============================================================
# LOSS
# ============================================================
criterion = nn.MSELoss()

def landmark_distance_mm(pred_world, gt_world):
    return float(np.linalg.norm(pred_world - gt_world))


# ============================================================
# TRAIN / VAL
# ============================================================
train_dataset = LandmarkDataset(train_data, LANDMARK_NAMES, TARGET_SHAPE, SIGMA)
val_dataset   = LandmarkDataset(val_data, LANDMARK_NAMES, TARGET_SHAPE, SIGMA)
test_dataset  = LandmarkDataset(test_data, LANDMARK_NAMES, TARGET_SHAPE, SIGMA)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = UNet3D(in_channels=1, out_channels=len(LANDMARK_NAMES), base=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val = 1e9
best_path = "best_landmark_model.pth"

for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for ct, heatmaps, _, _, _ in train_loader:
        ct = ct.to(device)                  # (B, 1, D, H, W)
        heatmaps = heatmaps.to(device)      # (B, L, D, H, W)

        pred = model(ct)
        loss = criterion(pred, heatmaps)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()

    train_loss = running / len(train_loader)

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for ct, heatmaps, _, _, _ in val_loader:
            ct = ct.to(device)
            heatmaps = heatmaps.to(device)
            pred = model(ct)
            loss = criterion(pred, heatmaps)
            val_running += loss.item()

    val_loss = val_running / len(val_loader)

    print(f"Epoch {epoch+1:03d} | train loss: {train_loss:.5f} | val loss: {val_loss:.5f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), best_path)

print("Best val loss:", best_val)


# ============================================================
# INFERENCE
# ============================================================
def predict_landmarks(model, ct_path, desc_path=None, target_shape=TARGET_SHAPE):
    ct_img = nib.load(ct_path)
    ct = ct_img.get_fdata().astype(np.float32)
    affine = ct_img.affine

    ct_net = np.transpose(ct, (2, 0, 1))
    ct_net = normalize_volume(ct_net)
    ct_tensor = resize_volume_3d(ct_net, target_shape).unsqueeze(0).to(device)  # (1,1,D,H,W)

    model.eval()
    with torch.no_grad():
        pred_hm = model(ct_tensor)[0].cpu()  # (L, D, H, W)

    pred_world = {}
    pred_vox = {}

    orig_net_shape = np.array([ct.shape[2], ct.shape[0], ct.shape[1]], dtype=np.float32)
    scale_back = (orig_net_shape - 1.0) / (np.array(target_shape, dtype=np.float32) - 1.0 + 1e-8)

    for i, name in enumerate(LANDMARK_NAMES):
        hm = pred_hm[i]
        coord_net = decode_heatmap_argmax(hm)   # in target net space

        # back to original net space
        coord_net_orig = coord_net * scale_back

        # back to original voxel xyz
        coord_xyz = net_to_voxel_xyz(coord_net_orig)
        pred_vox[name] = coord_xyz

        # to world
        world = apply_affine(affine, coord_xyz)
        pred_world[name] = world

    gt_world = None
    if desc_path is not None:
        desc = load_desc(desc_path)
        gt_world = desc["landmarks"]

    return pred_world, pred_vox, gt_world, pred_hm


# ============================================================
# EVALUATION
# ============================================================
def evaluate_split(data_list, model):
    all_errors = {name: [] for name in LANDMARK_NAMES}
    mean_errors = []

    for item in data_list:
        pred_world, _, gt_world, _ = predict_landmarks(model, item["ct"], item["desc"])

        subj_errors = []
        for name in LANDMARK_NAMES:
            err = landmark_distance_mm(pred_world[name], gt_world[name])
            all_errors[name].append(err)
            subj_errors.append(err)

        mean_errors.append(np.mean(subj_errors))

        print(f"\nSubject {item['subject']}")
        for name in LANDMARK_NAMES:
            print(f"  {name}: {all_errors[name][-1]:.2f} mm")

    print("\n================ SUMMARY ================")
    for name in LANDMARK_NAMES:
        print(f"{name}: mean error = {np.mean(all_errors[name]):.2f} mm")

    print(f"Overall mean landmark error = {np.mean(mean_errors):.2f} mm")


# ============================================================
# LOAD BEST MODEL AND TEST
# ============================================================
model.load_state_dict(torch.load(best_path, map_location=device))
model.to(device)

evaluate_split(test_data, model)

Train: 15
Val: 3
Test: 5
Epoch 001 | train loss: 0.05393 | val loss: 0.02453
Epoch 002 | train loss: 0.01951 | val loss: 0.00859
Epoch 003 | train loss: 0.01058 | val loss: 0.00770
Epoch 004 | train loss: 0.00692 | val loss: 0.00640
Epoch 005 | train loss: 0.00504 | val loss: 0.00470
Epoch 006 | train loss: 0.00392 | val loss: 0.00372
Epoch 007 | train loss: 0.00321 | val loss: 0.00304
Epoch 008 | train loss: 0.00268 | val loss: 0.00239
Epoch 009 | train loss: 0.00229 | val loss: 0.00205
Epoch 010 | train loss: 0.00201 | val loss: 0.00179
Epoch 011 | train loss: 0.00179 | val loss: 0.00159
Epoch 012 | train loss: 0.00161 | val loss: 0.00143
Epoch 013 | train loss: 0.00147 | val loss: 0.00127
Epoch 014 | train loss: 0.00135 | val loss: 0.00116
Epoch 015 | train loss: 0.00125 | val loss: 0.00105
Epoch 016 | train loss: 0.00116 | val loss: 0.00098
Epoch 017 | train loss: 0.00108 | val loss: 0.00090
Epoch 018 | train loss: 0.00102 | val loss: 0.00088
Epoch 019 | train loss: 0.00096 | val l

In [21]:
torch.save(model.state_dict(), "unet3d_final.pth")